In [ ]:
from pathlib import Path
import pickle
import numpy as np
import pandas as pd
import rasterio
from rasterio.features import shapes, rasterize
import geopandas as gpd
from shapely.geometry import shape, Point
from shapely.ops import unary_union
from scipy.ndimage import uniform_filter, mean as ndi_mean, standard_deviation as ndi_std
from skimage.segmentation import watershed, slic
from skimage.measure import regionprops, regionprops_table
from skimage.feature import peak_local_max
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

import importlib
ground = importlib.import_module('00_ground_truth_helpers')

In [ ]:
FILE_NUM = '03'

In [ ]:
# --- Phase 4.1: LiDAR crown detection ---
CHM_TREE_DETECT_HEIGHT = 1.5      # m, minimum height for tree top detection (training use only)
LOCAL_MAX_BASE_RADIUS = 0.5       # m, base of variable window
LOCAL_MAX_HEIGHT_COEF = 0.05      # window_radius = base + coef * height
CROWN_MIN_AREA = 1.0              # m^2
CROWN_MAX_HEIGHT_STD = 3.0        # m, drop merged multi-tree crowns
CROWN_MIN_MAX_HEIGHT = 2.0        # m, crown max height must exceed this for filter_passed

# --- Phase 4.2: SLIC segmentation at 1m ---
SLIC_N_SEGMENTS = 100_000         # ~10 pixels per segment on 1000x1000 tile
SLIC_COMPACTNESS = 10

# --- Phase 4.3: segment labeling ---
LABEL_TREE_MIN_CROWN_FRACTION = 0.5    # >50% of segment pixels inside a filter_passed crown
LABEL_NONTREE_CHM_MAX = 0.5            # m; CHM-only criterion (no NDVI constraint)
LABEL_NONTREE_MIN_FRACTION = 0.7       # >70% of segment pixels below CHM max

# --- Phase 4.7: tree union ---
CHM_TREE_OPERATIONAL_HEIGHT = 2.0      # m, operational tree/shrub boundary for the union
TREE_PROB_THRESHOLD = 0.7
RGB_HIGH_CONF_PROB = 0.85

RF_FEATURE_COLUMNS = ['r_mean', 'r_std', 'g_mean', 'g_std', 'b_mean', 'b_std', 'ndvi_mean', 'ndvi_std', 'savi_mean', 'savi_std', 'green_local_var_mean', 'ndvi_context_mean', 'area', 'perimeter', 'eccentricity', 'solidity', 'compactness']

GRID_SHAPE_1M = (1000, 1000)

# Functions

In [ ]:
def read_raster(path, out_shape, bands):
    """Read raster with explicit out_shape and bands. Returns array, native shape, native band count, and profile."""
    with rasterio.open(path) as src:
        native_shape = (src.height, src.width)
        native_band_count = src.count
        nodata = src.nodata
        profile = src.profile.copy()
        arr = src.read(bands, out_shape=(len(bands), out_shape[0], out_shape[1]))
    arr = arr.astype(np.float32)
    if nodata is not None:
        arr[arr == nodata] = np.nan
    return arr, native_shape, native_band_count, profile


def write_geotiff(array, reference_profile, out_path, dtype, nodata):
    """Write a 2D single-band array as GeoTIFF matching reference_profile."""
    prof = reference_profile.copy()
    prof.update(count=1, dtype=dtype, nodata=nodata, compress='lzw', height=array.shape[0], width=array.shape[1])
    with rasterio.open(out_path, 'w', **prof) as dst:
        dst.write(array.astype(dtype), 1)

In [ ]:
def detect_tree_tops(chm, min_height, base_radius, height_coef):
    """
    Detect tree tops in a CHM using a variable-radius local-maximum rule.

    radius(h) = base_radius + height_coef * h. Taller trees claim larger neighborhoods,
    preventing multiple peaks per crown, while shorter trees can still be detected between them.

    Inputs:
        chm         : 2D float array of canopy heights (m), 1m grid.
        min_height  : minimum peak height to be considered a tree.
        base_radius : base of the variable window (m == pixels at 1m).
        height_coef : slope of window radius vs. height.
    Outputs:
        tops : (N, 3) float array of [row, col, height].
    """
    chm_f = np.where(np.isfinite(chm), chm, -np.inf)
    local_maximum_coordinates = peak_local_max(chm_f, min_distance=1, threshold_abs=min_height, exclude_border=False) # returns local maximum
    if local_maximum_coordinates.size == 0:
        return np.empty((0, 3), dtype=np.float32)
    local_maximum_tree_heights = chm_f[local_maximum_coordinates[:, 0], local_maximum_coordinates[:, 1]] # get tree heights at local max coordinates
    order = np.argsort(-local_maximum_tree_heights) # return in descending order of tree height
    local_maximum_coordinates = local_maximum_coordinates[order]
    local_maximum_tree_heights = local_maximum_tree_heights[order]
    H, W = chm_f.shape
    claimed_tree_tops = np.zeros((H, W), dtype=bool)
    kept = []
    for (r, c), h in zip(local_maximum_coordinates, local_maximum_tree_heights):
        if claimed_tree_tops[r, c]: # already checked this treetop
            continue
        radius = int(np.ceil(base_radius + height_coef * h))
        r0, r1 = max(0, r - radius), min(H, r + radius + 1)
        c0, c1 = max(0, c - radius), min(W, c + radius + 1)
        rs, cs = np.ogrid[r0:r1, c0:c1] # create mesh grid
        circle = (rs - r) ** 2 + (cs - c) ** 2 <= radius ** 2
        claimed_tree_tops[r0:r1, c0:c1] |= circle # fill in tree top circle
        kept.append((r, c, h))
    return np.asarray(kept, dtype=np.float32)

In [ ]:
def segment_crowns_watershed(chm, tree_tops, min_height):
    """
    Delineate tree crowns via marker-controlled watershed on the inverted CHM.

    Inputs:
        chm        : 2D float CHM (m).
        tree_tops  : (N, 3) [row, col, height] tree tops.
        min_height : mask floor; pixels below this are excluded from any crown.
    Outputs:
        labels : 2D int32 array, 0 = non-crown, > 0 = tree ID.
    """
    H, W = chm.shape
    markers = np.zeros((H, W), dtype=np.int32)
    for i, (r, c, _) in enumerate(tree_tops, start=1):
        markers[int(r), int(c)] = i
    mask = np.isfinite(chm) & (chm > min_height)
    surface = -np.where(np.isfinite(chm), chm, 0)
    labels = watershed(surface, markers=markers, mask=mask)
    return labels.astype(np.int32)

In [ ]:
def build_crown_records(labels, chm, tree_tops, transform, crs, base_radius, height_coef, min_area, max_height_std, min_max_height):
    """
    Build per-crown circular geometry, peak points, and attributes recomputed from the circle footprint.

    Purpose:
        Replace the irregular watershed crown polygon with a circle centered at the tree-top peak,
        radius = base_radius + height_coef * peak_height (same formula as local-maxima detection).
        Height statistics and area are recomputed from CHM pixels falling inside the new circular
        footprint rather than the original watershed pixels, so geometry and attributes stay consistent.

    Inputs:
        labels         : 2D int32 watershed crown label raster (used only to confirm tid existence).
        chm            : 2D float CHM (m), 1m grid.
        tree_tops      : (N, 3) [row, col, height] tree tops.
        transform      : rasterio Affine for the grid.
        crs            : CRS of the grid.
        base_radius    : base of the radius formula (m).
        height_coef    : slope of the radius formula (m per m height).
        min_area       : min crown area (m^2) for filter_passed.
        max_height_std : max within-circle height std (m) for filter_passed.
        min_max_height : min within-circle max height (m) for filter_passed.
    Outputs:
        crowns_gdf : GeoDataFrame of circular crown polygons with recomputed attributes.
        peaks_gdf  : GeoDataFrame of tree top points with filter_passed matching crowns_gdf.
    """
    H, W = labels.shape
    px_size = transform.a
    crown_rows = []
    peak_rows = []
    for i, (r, c, h) in enumerate(tree_tops, start=1):
        radius_m = base_radius + height_coef * h
        px, py = rasterio.transform.xy(transform, r, c)
        circle = Point(px, py).buffer(radius_m)
        minx, miny, maxx, maxy = circle.bounds
        row_min, col_min = rasterio.transform.rowcol(transform, minx, maxy)
        row_max, col_max = rasterio.transform.rowcol(transform, maxx, miny)
        row_min = max(0, row_min)
        col_min = max(0, col_min)
        row_max = min(H, row_max + 1)
        col_max = min(W, col_max + 1)
        edge_clipped = bool(row_min == 0 or col_min == 0 or row_max == H or col_max == W)
        window_chm = chm[row_min:row_max, col_min:col_max]
        window_mask = rasterize([(circle, 1)], out_shape=window_chm.shape, transform=rasterio.transform.from_origin(transform.c + col_min * px_size, transform.f + row_min * transform.e, px_size, -transform.e), fill=0, dtype='uint8').astype(bool)
        chm_vals = window_chm[window_mask]
        chm_vals = chm_vals[np.isfinite(chm_vals)]
        max_h = float(np.max(chm_vals)) if chm_vals.size else 0.0
        mean_h = float(np.mean(chm_vals)) if chm_vals.size else 0.0
        h_std = float(np.std(chm_vals)) if chm_vals.size else 0.0
        area_m2 = float(circle.area)
        filter_passed = bool((area_m2 >= min_area) and (not edge_clipped) and (h_std <= max_height_std) and (max_h >= min_max_height))
        crown_rows.append({'tree_id': i, 'radius_m': radius_m, 'area_m2': area_m2, 'max_height': max_h, 'mean_height': mean_h, 'height_std': h_std, 'edge_clipped': edge_clipped, 'filter_passed': filter_passed, 'geometry': circle})
        peak_rows.append({'tree_id': i, 'height': float(h), 'filter_passed': filter_passed, 'geometry': Point(px, py)})
    crowns_gdf = gpd.GeoDataFrame(crown_rows, crs=crs)
    peaks_gdf = gpd.GeoDataFrame(peak_rows, crs=crs)
    return crowns_gdf, peaks_gdf

In [ ]:
def run_lidar_crowns_per_tile(tile_id):
    """Execute Phase 4.1 for one tile: local maxima -> watershed (detection only) -> circular crowns -> filter -> write GPKG."""
    _, _, _, chm_path = ground.build_paths(tile_id)
    print(f"[4.1] Tile {tile_id}")
    print(f"CHM: {chm_path.name}")
    chm_arr, chm_native, _, chm_profile = read_raster(chm_path, GRID_SHAPE_1M, [1])
    chm = chm_arr[0]
    print(f"CHM native {chm_native}, read {chm.shape}")
    tops = detect_tree_tops(chm, CHM_TREE_DETECT_HEIGHT, LOCAL_MAX_BASE_RADIUS, LOCAL_MAX_HEIGHT_COEF)
    print(f"detected tree tops: {tops.shape[0]}")
    labels = segment_crowns_watershed(chm, tops, CHM_TREE_DETECT_HEIGHT)
    n_crowns = int(labels.max())
    print(f"watershed crowns (detection only): {n_crowns}")
    crowns_gdf, peaks_gdf = build_crown_records(labels, chm, tops, chm_profile['transform'], chm_profile['crs'], LOCAL_MAX_BASE_RADIUS, LOCAL_MAX_HEIGHT_COEF, CROWN_MIN_AREA, CROWN_MAX_HEIGHT_STD, CROWN_MIN_MAX_HEIGHT)
    n_kept = int(crowns_gdf['filter_passed'].sum())
    print(f"crowns kept (filter_passed): {n_kept} / {len(crowns_gdf)}")
    peaks_path = ground.OUTPUT_DIR / f"{FILE_NUM}_tree_crown_peaks_{tile_id}_{ground.YEAR}.gpkg"
    outlines_path = ground.OUTPUT_DIR / f"{FILE_NUM}_tree_crown_outlines_{tile_id}_{ground.YEAR}.gpkg"
    peaks_gdf.to_file(peaks_path, driver='GPKG')
    crowns_gdf.to_file(outlines_path, driver='GPKG')
    print(f"wrote {peaks_path.name}")
    print(f"wrote {outlines_path.name}")
    return crowns_gdf, peaks_gdf, labels, chm_profile, chm

In [ ]:
def load_rgb_1m(rgb_path):
    """Read RGB downsampled to 1m grid; return (3, H, W) float."""
    rgb_arr, rgb_native, _, _ = read_raster(rgb_path, GRID_SHAPE_1M, [1, 2, 3])
    return rgb_arr, rgb_native


def slic_segment(rgb_1m, n_segments, compactness):
    """
    SLIC superpixel segmentation on 1m RGB.

    Inputs:
        rgb_1m      : (3, H, W) float RGB.
        n_segments  : target number of superpixels.
        compactness : SLIC compactness parameter.
    Outputs:
        segments : 2D int32 segment IDs (0 = nodata).
    """
    img = np.transpose(rgb_1m, (1, 2, 0)).astype(np.float32) / 255.0
    nan_mask = ~np.isfinite(img).all(axis=2)
    img = np.nan_to_num(img, nan=0.0)
    segments = slic(img, n_segments=n_segments, compactness=compactness, start_label=1, channel_axis=2)
    segments = segments.astype(np.int32)
    segments[nan_mask] = 0
    return segments

In [ ]:
def label_segments(segments, crowns_gdf, chm, transform, tree_min_crown_fraction, nontree_chm_max, nontree_min_fraction):
    """
    Assign 'tree', 'nontree', or 'ambiguous' to each SLIC segment.

    Tree      : >tree_min_crown_fraction of segment pixels inside a filter_passed crown.
    Non-tree  : >nontree_min_fraction of segment pixels have CHM < nontree_chm_max.
    Ambiguous : everything else.

    Inputs:
        segments                : 2D int32 SLIC label raster.
        crowns_gdf              : GeoDataFrame of crown polygons (uses filter_passed==True).
        chm                     : 2D float CHM.
        transform               : rasterio Affine.
        tree_min_crown_fraction : min fraction inside crown to label 'tree'.
        nontree_chm_max         : max CHM (m) for a pixel to count toward non-tree.
        nontree_min_fraction    : min fraction of low-CHM pixels to label 'nontree'.
    Outputs:
        label_map : dict {segment_id: 'tree' | 'nontree' | 'ambiguous'}.
    """
    H, W = segments.shape
    kept = crowns_gdf[crowns_gdf['filter_passed']]
    if len(kept) > 0:
        crown_mask = rasterize(((geom, 1) for geom in kept.geometry), out_shape=(H, W), transform=transform, fill=0, dtype='uint8').astype(bool)
    else:
        crown_mask = np.zeros((H, W), dtype=bool)
    chm_f = np.where(np.isfinite(chm), chm, 999)
    nontree_pixel = chm_f < nontree_chm_max
    valid = segments > 0
    seg_flat = segments.ravel()
    valid_flat = valid.ravel()
    n_seg = int(segments.max()) + 1
    total = np.bincount(seg_flat, weights=valid_flat.astype(np.float64), minlength=n_seg)
    tree_pix = np.bincount(seg_flat, weights=(crown_mask & valid).ravel().astype(np.float64), minlength=n_seg)
    nontree_pix = np.bincount(seg_flat, weights=(nontree_pixel & valid).ravel().astype(np.float64), minlength=n_seg)
    label_map = {}
    for sid in range(1, n_seg):
        n = total[sid]
        if n == 0:
            continue
        tree_frac = tree_pix[sid] / n
        nontree_frac = nontree_pix[sid] / n
        if tree_frac > tree_min_crown_fraction:
            label_map[sid] = 'tree'
        elif nontree_frac > nontree_min_fraction:
            label_map[sid] = 'nontree'
        else:
            label_map[sid] = 'ambiguous'
    return label_map

In [ ]:
def extract_segment_features(segments, rgb_1m, ndvi, savi):
    """
    Per-segment features from RGB + NDVI + SAVI (transferable to NAIP).

    Spectral : mean/std of R, G, B, NDVI, SAVI.
    Texture  : mean of 3x3 local variance of green.
    Shape    : area, perimeter, eccentricity, solidity, compactness.
    Context  : mean of 5x5 local NDVI.

    NaN feature values (from empty/invalid segments) are filled with 0.

    Inputs:
        segments : 2D int32 SLIC label raster.
        rgb_1m   : (3, H, W) RGB at 1m.
        ndvi     : 2D float NDVI at 1m.
        savi     : 2D float SAVI at 1m.
    Outputs:
        df : pandas.DataFrame indexed by segment_id, all features filled to finite values.
    """
    seg_ids = np.unique(segments)
    seg_ids = seg_ids[seg_ids > 0]
    r_ch = np.nan_to_num(rgb_1m[0], nan=0.0)
    g_ch = np.nan_to_num(rgb_1m[1], nan=0.0)
    b_ch = np.nan_to_num(rgb_1m[2], nan=0.0)
    ndvi_f = np.nan_to_num(ndvi, nan=0.0)
    savi_f = np.nan_to_num(savi, nan=0.0)
    feats = {'segment_id': seg_ids}
    for name, arr in [('r', r_ch), ('g', g_ch), ('b', b_ch), ('ndvi', ndvi_f), ('savi', savi_f)]:
        feats[f'{name}_mean'] = ndi_mean(arr, labels=segments, index=seg_ids)
        feats[f'{name}_std']  = ndi_std(arr, labels=segments, index=seg_ids)
    local_var_g = uniform_filter(g_ch ** 2, size=3) - uniform_filter(g_ch, size=3) ** 2
    feats['green_local_var_mean'] = ndi_mean(local_var_g, labels=segments, index=seg_ids)
    local_ndvi = uniform_filter(ndvi_f, size=5)
    feats['ndvi_context_mean'] = ndi_mean(local_ndvi, labels=segments, index=seg_ids)
    props = regionprops_table(segments, properties=('label', 'area', 'perimeter', 'eccentricity', 'solidity'))
    shape_df = pd.DataFrame(props).set_index('label').reindex(seg_ids)
    shape_df['compactness'] = 4 * np.pi * shape_df['area'] / (shape_df['perimeter'] ** 2 + 1e-6)
    for col in ['area', 'perimeter', 'eccentricity', 'solidity', 'compactness']:
        feats[col] = shape_df[col].values
    df = pd.DataFrame(feats).set_index('segment_id')
    df = df.fillna(0.0)
    return df

In [ ]:
def process_tile_rgb(tile_id, crowns_gdf, chm_profile, chm_arr):
    """
    SLIC + labeling + feature extraction on one tile.

    Returns per-segment features_df, label_map, segments raster, and 1m RGB/NDVI/SAVI arrays.
    """
    rgb_path, ndvi_path, savi_path, _ = ground.build_paths(tile_id)
    print(f"[4.2-4.4] Tile {tile_id}")
    rgb_1m, rgb_native = load_rgb_1m(rgb_path)
    print(f"RGB native {rgb_native}, read {rgb_1m.shape[1:]}")
    ndvi_arr, _, _, _ = read_raster(ndvi_path, GRID_SHAPE_1M, [1])
    savi_arr, _, _, _ = read_raster(savi_path, GRID_SHAPE_1M, [1])
    ndvi = ndvi_arr[0]
    savi = savi_arr[0]
    segments = slic_segment(rgb_1m, SLIC_N_SEGMENTS, SLIC_COMPACTNESS)
    n_seg = int(segments.max())
    print(f"SLIC segments: {n_seg}")
    label_map = label_segments(segments, crowns_gdf, chm_arr, chm_profile['transform'], LABEL_TREE_MIN_CROWN_FRACTION, LABEL_NONTREE_CHM_MAX, LABEL_NONTREE_MIN_FRACTION)
    counts = pd.Series(list(label_map.values())).value_counts().to_dict()
    print(f"segment labels: {counts}")
    features_df = extract_segment_features(segments, rgb_1m, ndvi, savi)
    features_df['label'] = features_df.index.map(label_map)
    features_df['tile_id'] = tile_id
    print(f"feature rows: {len(features_df)}")
    return features_df, label_map, segments, rgb_1m, ndvi, savi

In [ ]:
def loo_cross_validate(all_features, feature_columns):
    """Leave-one-tile-out CV using only 'tree' and 'nontree' segments."""
    train_df = all_features[all_features['label'].isin(['tree', 'nontree'])].copy()
    y_true_all, y_pred_all = [], []
    for tile_id in train_df['tile_id'].unique():
        train = train_df[train_df['tile_id'] != tile_id]
        test = train_df[train_df['tile_id'] == tile_id]
        Xtr, ytr = train[feature_columns].values, train['label'].values
        Xte, yte = test[feature_columns].values, test['label'].values
        clf = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42, class_weight='balanced')
        clf.fit(Xtr, ytr)
        yhat = clf.predict(Xte)
        print(f"Fold: held-out tile {tile_id} (n_train={len(train)}, n_test={len(test)})")
        print(classification_report(yte, yhat, zero_division=0))
        y_true_all.extend(yte)
        y_pred_all.extend(yhat)
    print("=== Pooled LOO confusion matrix ===")
    print(confusion_matrix(y_true_all, y_pred_all, labels=['nontree', 'tree']))


def train_final_model(all_features, feature_columns):
    """Train final RF on all tree/nontree segments; save to OUTPUT_DIR as pickle."""
    train_df = all_features[all_features['label'].isin(['tree', 'nontree'])]
    X, y = train_df[feature_columns].values, train_df['label'].values
    clf = RandomForestClassifier(n_estimators=500, n_jobs=-1, random_state=42, class_weight='balanced')
    clf.fit(X, y)
    model_path = ground.OUTPUT_DIR / f"{FILE_NUM}_rgb_tree_detector_{ground.YEAR}.pkl"
    with open(model_path, 'wb') as f:
        pickle.dump({'model': clf, 'feature_columns': feature_columns}, f)
    print(f"saved model: {model_path.name}")
    print("feature importances:")
    for name, imp in sorted(zip(feature_columns, clf.feature_importances_), key=lambda t: -t[1]):
        print(f"{name:25s} {imp:.4f}")
    return clf

In [ ]:
def tree_outputs_per_tile(tile_id, clf, features_df, segments, chm_arr, chm_profile, feature_columns, chm_operational_height, prob_threshold, rgb_high_conf_prob):
    """
    Predict tree prob per SLIC segment, union with CHM > chm_operational_height, write outputs.

    Tree mask = (CHM > chm_operational_height) UNION (RGB prob > prob_threshold).
    Confidence: High = both sources agree; Medium = CHM-only OR RGB prob > rgb_high_conf_prob;
                Low = RGB-only with prob in [prob_threshold, rgb_high_conf_prob].
    """
    tile_feats = features_df[features_df['tile_id'] == tile_id].copy()
    X = tile_feats[feature_columns].values
    proba = clf.predict_proba(X)
    tree_idx = list(clf.classes_).index('tree')
    tile_feats['tree_prob'] = proba[:, tree_idx]
    prob_map = np.full(segments.shape, np.nan, dtype=np.float32)
    prob_lookup = tile_feats['tree_prob'].to_dict()
    seg_ids = np.unique(segments)
    seg_ids = seg_ids[seg_ids > 0]
    lookup_arr = np.full(int(seg_ids.max()) + 1, np.nan, dtype=np.float32)
    for sid, p in prob_lookup.items():
        lookup_arr[int(sid)] = p
    valid = segments > 0
    prob_map[valid] = lookup_arr[segments[valid]]
    chm_tree = np.isfinite(chm_arr) & (chm_arr > chm_operational_height)
    rgb_tree = np.isfinite(prob_map) & (prob_map > prob_threshold)
    tree_mask = (chm_tree | rgb_tree).astype(np.uint8)
    confidence = np.zeros(segments.shape, dtype=np.uint8)
    both = chm_tree & rgb_tree
    chm_only = chm_tree & ~rgb_tree
    rgb_high = rgb_tree & (prob_map > rgb_high_conf_prob) & ~chm_tree
    rgb_low = rgb_tree & (prob_map <= rgb_high_conf_prob) & ~chm_tree
    confidence[rgb_low] = 1
    confidence[chm_only | rgb_high] = 2
    confidence[both] = 3
    mask_path = ground.OUTPUT_DIR / f"{FILE_NUM}_tree_mask_{tile_id}_{ground.YEAR}.tif"
    conf_path = ground.OUTPUT_DIR / f"{FILE_NUM}_tree_confidence_{tile_id}_{ground.YEAR}.tif"
    prob_path = ground.OUTPUT_DIR / f"{FILE_NUM}_tree_segment_prob_{tile_id}_{ground.YEAR}.tif"
    write_geotiff(tree_mask, chm_profile, mask_path, 'uint8', 255)
    write_geotiff(confidence, chm_profile, conf_path, 'uint8', 255)
    write_geotiff(np.where(np.isfinite(prob_map), prob_map, -1).astype(np.float32), chm_profile, prob_path, 'float32', -1)
    n_tree = int(tree_mask.sum())
    n_valid = int((tree_mask != 255).sum())
    print(f"[{tile_id}] tree pixels: {n_tree:,} / {n_valid:,} ({n_tree / n_valid * 100:.2f}%)")
    print(f"[{tile_id}] wrote {mask_path.name}, {conf_path.name}, {prob_path.name}")

# Run

In [ ]:
print(f"=== Phase 4: tree detection ({ground.SITE_ID} {ground.YEAR}, {len(ground.TILE_IDS)} tiles) ===")
tile_state = {}
for tid in ground.TILE_IDS:
    crowns_gdf, peaks_gdf, labels, chm_profile, chm = run_lidar_crowns_per_tile(tid)
    tile_state[tid] = dict(crowns=crowns_gdf, chm_profile=chm_profile, chm=chm)

In [ ]:
all_features = []
seg_by_tile = {}
for tid in ground.TILE_IDS:
    feats, label_map, segments, rgb_1m, ndvi, savi = process_tile_rgb(tid, tile_state[tid]['crowns'], tile_state[tid]['chm_profile'], tile_state[tid]['chm'])
    all_features.append(feats)
    seg_by_tile[tid] = segments
all_features = pd.concat(all_features, axis=0)

In [ ]:
print("=== 4.5 LOO cross-validation ===")
loo_cross_validate(all_features, RF_FEATURE_COLUMNS)
print("=== 4.5 Final model training on pooled labels ===")
clf = train_final_model(all_features, RF_FEATURE_COLUMNS)

In [ ]:
print("=== 4.6-4.8 Predict + union + confidence ===")
for tid in ground.TILE_IDS:
    tree_outputs_per_tile(tid, clf, all_features, seg_by_tile[tid], tile_state[tid]['chm'], tile_state[tid]['chm_profile'], RF_FEATURE_COLUMNS, CHM_TREE_OPERATIONAL_HEIGHT, TREE_PROB_THRESHOLD, RGB_HIGH_CONF_PROB)